In [1]:
!apt-get -qq update
!apt-get -qq install -y openbabel

!obabel -V
!obabel -L charges
!obabel -L forcefields

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libboost-iostreams1.74.0:amd64.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../libboost-iostreams1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-iostreams1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libinchi1.
Preparing to unpack .../libinchi1_1.03+dfsg-4_amd64.deb ...
Unpacking libinchi1 (1.03+dfsg-4) ...
Selecting previously unselected package libmaeparser1:amd64.
Preparing to unpack .../libmaeparser1_1.2.4-1build1_amd64.deb ...
Unpacking libmaeparser1:amd64 (1.2.4-1build1) ...
Selecting previously unselected package libopenbabel7.
Preparing to unpack .../libopenbabel7_3.1.1+dfsg-6ubuntu5_amd64.deb ...
Unpacking libopenbabel7 (3.1.1+dfsg-6ubuntu5) ...
Selecting previousl

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from pathlib import Path


INPUT_DIR = Path("/content/drive/MyDrive/farmacos/Farmaco/TP FINAL/INSECTICIDAS")

# Carpeta de salida
OUTPUT_DIR = Path("/content/drive/MyDrive/farmacos/Farmaco/TP FINAL/INSECTICIDAS/ligandos_preparados")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta de entrada:", INPUT_DIR)
print("Carpeta de salida:", OUTPUT_DIR)

Carpeta de entrada: /content/drive/MyDrive/farmacos/Farmaco/TP FINAL/INSECTICIDAS
Carpeta de salida: /content/drive/MyDrive/farmacos/Farmaco/TP FINAL/INSECTICIDAS/ligandos_preparados


In [8]:
import subprocess
import shutil
import re
from pathlib import Path
import pandas as pd


FORCEFIELD_PRIMARY = "MMFF94"
FORCEFIELD_FALLBACK = "UFF"
STEPS = 2000

ALLOWED_EXTENSIONS = [".sdf", ".mol2", ".mol"]

FORCE_GEN3D = False


# =========================
# FUNCIONES AUXILIARES
# =========================

def clean_name(name):
    """
    Limpia nombres de archivo
    """
    name = name.strip()
    name = re.sub(r"[^A-Za-z0-9_\-]+", "_", name)
    return name


def run_obabel(cmd):
    """
    Ejecuta un comando de Open Babel
    """
    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(
            "Falló Open Babel.\n"
            f"Comando:\n{' '.join(cmd)}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )

    return result.stdout, result.stderr


def count_pdbqt_atoms(pdbqt_file):
    """
    Cuenta átomos en un archivo PDBQT.
    """
    n_atoms = 0
    with open(pdbqt_file, "r", errors="ignore") as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                n_atoms += 1
    return n_atoms


def estimate_total_charge_from_pdbqt(pdbqt_file):
    """
    Estima la carga total sumando las cargas parciales del PDBQT.
    En PDBQT, la carga suele estar en la anteúltima columna al hacer split().
    """
    charges = []
    with open(pdbqt_file, "r", errors="ignore") as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                parts = line.split()
                try:
                    charge = float(parts[-2])
                    charges.append(charge)
                except:
                    pass

    if len(charges) == 0:
        return None

    return sum(charges)


#buscamos ligandos

ligand_files = []

for ext in ALLOWED_EXTENSIONS:
    ligand_files.extend(INPUT_DIR.glob(f"*{ext}"))
    ligand_files.extend(INPUT_DIR.glob(f"*{ext.upper()}"))

ligand_files = sorted(set(ligand_files))

if len(ligand_files) == 0:
    raise FileNotFoundError(
        f"No encontré archivos SDF/MOL2/MOL en:\n{INPUT_DIR}\n"
    )

print(f"ligandos:")
for f in ligand_files:
    print(" -", f.name)

ligandos:
 - 12Bromododecanol.sdf
 - Bombykol.sdf
 - Limoneno.sdf
 - Nicotina.sdf
 - Rhodojaponina.sdf


In [9]:
results = []

for ligand_path in ligand_files:
    ligand_name = clean_name(ligand_path.stem)

    print("\n" + "="*70)
    print(f"Procesando ligando: {ligand_path.name}")
    print("="*70)

    # Archivos intermedios y finales
    h_sdf = OUTPUT_DIR / f"{ligand_name}_01_H.sdf"
    opt_sdf = OUTPUT_DIR / f"{ligand_name}_02_H_min.sdf"
    charged_mol2 = OUTPUT_DIR / f"{ligand_name}_03_H_min_gasteiger.mol2"
    pdbqt_file = OUTPUT_DIR / f"{ligand_name}.pdbqt"

    try:
        # agregamos hidrógenos
        cmd_h = [
            "obabel",
            str(ligand_path),
            "-O", str(h_sdf),
            "-h"
        ]

        if FORCE_GEN3D:
            cmd_h.append("--gen3d")

        run_obabel(cmd_h)
        print(" hidrógenos explícitos agregados.")

        # optimizamos geometría
        cmd_min = [
            "obabel",
            str(h_sdf),
            "-O", str(opt_sdf),
            "--minimize",
            "--ff", FORCEFIELD_PRIMARY,
            "--steps", str(STEPS),
            "--sd"
        ]

        try:
            run_obabel(cmd_min)
            used_forcefield = FORCEFIELD_PRIMARY
        except Exception as e_primary:
            print(f"MMFF94 falló para {ligand_name}. Intento con UFF...")

            cmd_min_fallback = [
                "obabel",
                str(h_sdf),
                "-O", str(opt_sdf),
                "--minimize",
                "--ff", FORCEFIELD_FALLBACK,
                "--steps", str(STEPS),
                "--sd"
            ]
            run_obabel(cmd_min_fallback)
            used_forcefield = FORCEFIELD_FALLBACK

        print(f"geometría optimizada con {used_forcefield}.")

        # calcular cargas
        cmd_charge_mol2 = [
            "obabel",
            str(opt_sdf),
            "-O", str(charged_mol2),
            "--partialcharge", "gasteiger"
        ]
        run_obabel(cmd_charge_mol2)
        print("cargas Gasteiger calculadas y guardadas en MOL2.")

        #convertir a pdbqt
        cmd_pdbqt = [
            "obabel",
            str(opt_sdf),
            "-O", str(pdbqt_file),
            "--partialcharge", "gasteiger",
            "-xh"
        ]
        run_obabel(cmd_pdbqt)
        print("ligando convertido a PDBQT para AutoDock Vina.")

        # Validación simple
        n_atoms = count_pdbqt_atoms(pdbqt_file)
        total_charge = estimate_total_charge_from_pdbqt(pdbqt_file)

        print(f"Validación: {n_atoms} átomos en PDBQT.")
        if total_charge is not None:
            print(f"Carga total aproximada en PDBQT: {total_charge:.3f}")

        results.append({
            "ligando": ligand_path.name,
            "estado": "OK",
            "forcefield": used_forcefield,
            "archivo_H": h_sdf.name,
            "archivo_optimizado": opt_sdf.name,
            "archivo_cargas_mol2": charged_mol2.name,
            "archivo_pdbqt": pdbqt_file.name,
            "atomos_pdbqt": n_atoms,
            "carga_total_aprox": total_charge
        })

    except Exception as e:
        print("Falló este ligando.")
        print(e)

        results.append({
            "ligando": ligand_path.name,
            "estado": "ERROR",
            "forcefield": None,
            "archivo_H": None,
            "archivo_optimizado": None,
            "archivo_cargas_mol2": None,
            "archivo_pdbqt": None,
            "atomos_pdbqt": None,
            "carga_total_aprox": None,
            "error": str(e)
        })


# Tabla resumen
df_results = pd.DataFrame(results)
df_results


Procesando ligando: 12Bromododecanol.sdf
 hidrógenos explícitos agregados.
geometría optimizada con MMFF94.
cargas Gasteiger calculadas y guardadas en MOL2.
ligando convertido a PDBQT para AutoDock Vina.
Validación: 39 átomos en PDBQT.
Carga total aproximada en PDBQT: 0.007

Procesando ligando: Bombykol.sdf
 hidrógenos explícitos agregados.
geometría optimizada con MMFF94.
cargas Gasteiger calculadas y guardadas en MOL2.
ligando convertido a PDBQT para AutoDock Vina.
Validación: 47 átomos en PDBQT.
Carga total aproximada en PDBQT: 0.006

Procesando ligando: Limoneno.sdf
 hidrógenos explícitos agregados.
geometría optimizada con MMFF94.
cargas Gasteiger calculadas y guardadas en MOL2.
ligando convertido a PDBQT para AutoDock Vina.
Validación: 26 átomos en PDBQT.
Carga total aproximada en PDBQT: -0.004

Procesando ligando: Nicotina.sdf
 hidrógenos explícitos agregados.
geometría optimizada con MMFF94.
cargas Gasteiger calculadas y guardadas en MOL2.
ligando convertido a PDBQT para AutoD

,ligando,estado,forcefield,archivo_H,archivo_optimizado,archivo_cargas_mol2,archivo_pdbqt,atomos_pdbqt,carga_total_aprox
0,12Bromododecanol.sdf,OK,MMFF94,12Bromododecanol_01_H.sdf,12Bromododecanol_02_H_min.sdf,12Bromododecanol_03_H_min_gasteiger.mol2,12Bromododecanol.pdbqt,39,0.007
1,Bombykol.sdf,OK,MMFF94,Bombykol_01_H.sdf,Bombykol_02_H_min.sdf,Bombykol_03_H_min_gasteiger.mol2,Bombykol.pdbqt,47,0.006
2,Limoneno.sdf,OK,MMFF94,Limoneno_01_H.sdf,Limoneno_02_H_min.sdf,Limoneno_03_H_min_gasteiger.mol2,Limoneno.pdbqt,26,-0.004
3,Nicotina.sdf,OK,MMFF94,Nicotina_01_H.sdf,Nicotina_02_H_min.sdf,Nicotina_03_H_min_gasteiger.mol2,Nicotina.pdbqt,26,0.001
4,Rhodojaponina.sdf,OK,MMFF94,Rhodojaponina_01_H.sdf,Rhodojaponina_02_H_min.sdf,Rhodojaponina_03_H_min_gasteiger.mol2,Rhodojaponina.pdbqt,58,0.001
